# Analyzing United States Immigration and Customs Enforcement (ICE) Arrests from September 2023 to October 2025

Rin Hwang

## Introduction

In this project, I wanted to investigate U.S. Immigration and Customs Enforcement (ICE) data, my main question being: "How do the age distributions of ICE arrests vary across the top ten countries of citizenship and the states with the highest apprehension between 2023 and 2025?" 
The purpose of this analysis is to identify demographic and geographic patterns within ICE enforcement actions. By examining the intersection of age, nationality, and location, this study seeks to determine if enforcement trends differ significantly depending on where an arrest occurs or the citizenship country of the individual involved. Understanding these distributions provides critical insight into the demographic impact of immigration policy during this particular period.

In this dataset I chose from The Deportation Data Project, it records ICE arrests from September 2023 to mid-October 2025, and includes over 300,000 rows, some that are duplicated. This dataset is uniquely appropriate for this research because it provides person-level records of administrative arrests, containing specific and easy-to-read identifiers for dates, geographic locations, and birth years, allowing for a more precise calculation of age at the time of apprehension. 

I used the following variables from the dataset in order to effectively examine my research question:  

- apprehension_date: This variable records the exact date the individual was taken into custody. It is used to for calculating the age of the individual at the time of the event.

- birth_year: This variable represents the year the individual was born. By subtracting this from the year of the apprehension_date variable, we create a new calculated variable: age_at_apprehension.

- citizenship_country: This categorical variable identifies the individual's country of origin. This is used to filter the dataset to the top 5 most frequent nationalities. 

- apprehension_state: This variable indicates the U.S. state where the arrest took place. It allows for the grouping of data by region to compare if certain states show different age profiles in their enforcement statistics.

## Data Analysis

To address the research question using data analysis, I used three primary data wrangling techniques: data cleaning, data transformation, and grouping and aggregation. The analysis begins with cleaning rows with missing birth_year, citizenship_country, and apprehension_state to identify and remove any data that would affect the analysis.

Following this, I will perform a data transformation to create the "Age at Apprehension" (variable is age_at_apprenhension) by subtracting the birth year from the year component of the apprehension date. I will also use pd.cut to categorize these ages into five distinct age categories. Below is how I grouped the ages: 

Age groups (age by the time of arrest): 
- 0-18 years old (agegroup_1)
- 19-35 years old (agegroup_2)
- 36-59 years old (agegroup_3)
- 60-80 years old (agegroup_4)
- 81-100+ years old (agegroup_5)

Lastly, I will use grouping and aggregation to identify the top ten states and their top age group, top ten countries by arrests and their top age group and while also calculating the mean and median age for each. This approach allows for a critical comparison that reveals whether specific nationalities or geographic regions skew toward younger or older demographics, providing an explanation to how age distributions vary across the ICE enforcement.

### Importing libraries and dataset

In [16]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:/Users/hwang/OneDrive/Documents/MC stuff/Spring 2026/DATA 101 Introduction to Data Science/Projects/Project 1/arrests-latest.xlsx")

### Getting variable info

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 377067 entries, 0 to 377066
Data columns (total 25 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   apprehension_date           377067 non-null  datetime64[ns]
 1   apprehension_state          316913 non-null  object        
 2   apprehension_aor            371180 non-null  object        
 3   final_program               377067 non-null  object        
 4   final_program_group         377067 non-null  object        
 5   apprehension_method         377067 non-null  object        
 6   apprehension_criminality    377067 non-null  object        
 7   case_status                 367695 non-null  object        
 8   case_category               367694 non-null  object        
 9   departed_date               236138 non-null  datetime64[ns]
 10  departure_country           236084 non-null  object        
 11  final_order_yes_no          367695 non-

### Data cleaning

In [29]:
df_clean = df.dropna(subset=['citizenship_country', 'apprehension_state', 'birth_year']).copy()

df_clean.loc[:, 'apprehension_date'] = pd.to_datetime(df_clean['apprehension_date'])
df_clean.loc[:, 'age_at_apprehension'] = df_clean['apprehension_date'].dt.year - df_clean['birth_year']

In [30]:
new = df[['apprehension_date','apprehension_state','citizenship_country','birth_year']]
print(new)

       apprehension_date     apprehension_state citizenship_country  \
0             2025-09-29             CALIFORNIA            HONDURAS   
1             2025-07-01  ARMED FORCES - EUROPE         AFGHANISTAN   
2             2025-10-10             CALIFORNIA             TURKIYE   
3             2025-10-08             CALIFORNIA             MOROCCO   
4             2025-09-30                  TEXAS              MEXICO   
...                  ...                    ...                 ...   
377062        2025-03-10              MINNESOTA             GERMANY   
377063        2024-04-08                    NaN              MEXICO   
377064        2023-12-12                    NaN              MEXICO   
377065        2024-04-01                    NaN              MEXICO   
377066        2025-08-24               COLORADO              MEXICO   

        birth_year  
0           2025.0  
1           2025.0  
2           2024.0  
3           2024.0  
4           2024.0  
...            ...  


### Data transformation

#### Creating age_of_apprehension variable

In [31]:
df['apprehension_date'] = pd.to_datetime(df['apprehension_date'])

df['age_at_apprehension'] = df['apprehension_date'].dt.year - df['birth_year']

print(df[['apprehension_date', 'birth_year', 'age_at_apprehension']].head())

  apprehension_date  birth_year  age_at_apprehension
0        2025-09-29      2025.0                  0.0
1        2025-07-01      2025.0                  0.0
2        2025-10-10      2024.0                  1.0
3        2025-10-08      2024.0                  1.0
4        2025-09-30      2024.0                  1.0


#### Creating age groups using bins

In [33]:
bins = [-1, 18, 35, 59, 80, np.inf]

labels = ['agegroup_1', 'agegroup_2', 'agegroup_3', 'agegroup_4', 'agegroup_5']

df['age_group'] = pd.cut(df['age_at_apprehension'], bins=bins, labels=labels)

print(df['age_group'].value_counts().sort_index())

age_group
agegroup_1     13033
agegroup_2    201079
agegroup_3    155569
agegroup_4      7346
agegroup_5        39
Name: count, dtype: int64


### Grouping and Aggregation

#### Identifying the top 10 citizenship countries by arrest and their top age group

In [47]:
top_10_countries = df['citizenship_country'].value_counts().nlargest(10).index

df_top_countries = df[df['citizenship_country'].isin(top_10_countries)]

country_summary = df_top_countries.groupby('citizenship_country')['age_group'].describe()

print(country_summary[['top']])

                            top
citizenship_country            
COLOMBIA             agegroup_2
CUBA                 agegroup_3
DOMINICAN REPUBLIC   agegroup_2
ECUADOR              agegroup_2
EL SALVADOR          agegroup_2
GUATEMALA            agegroup_2
HONDURAS             agegroup_2
MEXICO               agegroup_3
NICARAGUA            agegroup_2
VENEZUELA            agegroup_2


#### Identifying the top 10 apprehension states and their top age group

In [45]:
top_10_states = df['apprehension_state'].value_counts().nlargest(10).index

df_filtered = df[df['apprehension_state'].isin(top_10_states)]

state_summary = df_filtered.groupby('apprehension_state')['age_group'].describe()

print(state_summary[['top']])

                           top
apprehension_state            
ARIZONA             agegroup_2
CALIFORNIA          agegroup_3
FLORIDA             agegroup_2
GEORGIA             agegroup_2
LOUISIANA           agegroup_2
NEW JERSEY          agegroup_2
NEW YORK            agegroup_2
TENNESSEE           agegroup_2
TEXAS               agegroup_2
VIRGINIA            agegroup_2


#### Identifying how many each age group has and which has the most

In [44]:
print(df_clean['age_group'].value_counts())

highest_group = df_clean['age_group'].value_counts().idxmax()
print(f"The age group with the most arrests is: {highest_group}")

age_group
agegroup_2    168215
agegroup_3    132173
agegroup_1     10250
agegroup_4      6250
agegroup_5        24
Name: count, dtype: int64
The age group with the most arrests is: agegroup_2


#### Top 10 states with the oldest age by mean

In [40]:
state_age_stats = df_clean.groupby('apprehension_state')['age_at_apprehension'].agg(['mean', 'median', 'count'])

# Filtering out outliers
state_age_stats = state_age_stats[state_age_stats['count'] > 50]

print("Average Age of Apprehension by State (Sorted by Mean Age):")
print(state_age_stats.sort_values(by='mean', ascending=False).head(10))

Average Age of Apprehension by State (Sorted by Mean Age):
                                  mean  median  count
apprehension_state                                   
NORTHERN MARIANA ISLANDS     46.373832    47.0    107
GUAM                         40.654545    40.0     55
PUERTO RICO                  40.438849    39.0    973
ARMED FORCES - THE AMERICAS  39.807692    37.0    104
HAWAII                       39.196154    37.0    260
CALIFORNIA                   38.243507    38.0  25876
NEVADA                       36.636670    36.0   2967
ALASKA                       36.602273    37.0     88
VIRGIN ISLANDS               36.380753    35.0    239
DISTRICT OF COLUMBIA         36.269140    35.0   1267


### Summary Table of Top Apprehension States and their Statistics

In [55]:
summary_custom = df_clean.groupby('apprehension_state')['age_at_apprehension'].agg(
    Arrests='count',
    Average_Age='mean',
    Median_Age='median',
    Min_Age='min',
    Max_Age='max'
)

final_table = summary_custom.sort_values(by='Arrests', ascending=False).head(10)

print(final_table.round(1))

                    Arrests  Average_Age  Median_Age  Min_Age  Max_Age
apprehension_state                                                    
TEXAS                 81030         34.9        34.0      1.0     83.0
FLORIDA               30346         34.5        33.0      1.0     89.0
CALIFORNIA            25876         38.2        38.0      0.0     88.0
NEW YORK              13944         32.2        32.0      1.0     78.0
GEORGIA               11213         34.9        34.0      0.0     76.0
NEW JERSEY             9413         34.9        34.0      1.0     80.0
TENNESSEE              8802         32.9        32.0      0.0     79.0
ARIZONA                8750         34.3        33.0      2.0     79.0
VIRGINIA               8736         33.7        32.0      1.0     79.0
LOUISIANA              6873         35.4        34.0      2.0     76.0


## Conclusion and Future Directions

The data analysis of ICE arrest data from 2023 to 2025 reveals that while "agegroup_2" (19-35 years old) seem most targeted, significant variations exist when disaggregated by nationality and geography. For the majority of the top-arrested countries, such as those from Venezuela, Honduras, and Colombia, the data shows a heavy concentration in "agegroup_2" (19-35), likely reflecting recent migration trends and a focus on working-age individuals. However, Mexico and Cuba show that the most frequent age group shifts older to "agegroup_3" (36–59). This suggests that enforcement actions involving Mexican and Cuban nationals may involve individuals with longer-term residency or established community ties, contrasting with the younger individuals of newer migrant cohorts. 

The data shows a big split between border regions and non-border states. In Texas and Arizona, the focus is mostly on "agegroup_2" (19-35). California is the big exception here because even though it has a ton of arrests, most of its arrests involve people in "agegroup_3" (36–59). When looking at U.S. territories like Guam, the average age is even higher. This reveals that as enforcement moves away from the border and into more established communities, the people being arrested tend to be older.

These results are meaningful because they show that ICE doesn’t just do the same arresting everywhere. Instead, they seem to adjust based on the local area and politics. For instance, seeing more older people arrested in California and among Mexican nationals suggests that ICE there is likely arresting those who have lived in the U.S. for decades. On the other hand, the younger crowd being arrested in Texas shows a focus on the border and local jails. This distinction matters because it helps aid organizations figure out what kind of help to provide, since a young individual alone needs significantly different support than an older person with a family. 

To build upon this work, future research could look at "Years of Residency" (how long people have lived in the U.S.) and "Arrest Type" (how exactly they were arrested). Comparing an individual's age with how long they've been in the country would help show if these arrests are really targeting people who have been here for a long time. It would also be interesting to see if certain age groups are being singled out because of their criminal records, or if the age differences are just because some local police departments work more closely with ICE than others.

## References

Deportation Data Project. (2025). Immigration and Customs Enforcement – Deportation Data Project. Deportationdata.org. https://deportationdata.org/data/ice.html

UC Berkeley - The Goldman School Public Policy . (2026). ICE Arrest Trends: New Research Reveals Shifting Enforcement Patterns | Recent News | News Center | Research and Impact | Goldman School of Public Policy | University of California, Berkeley. Berkeley.edu. https://gspp.berkeley.edu/research-and-impact/news/recent-news/ice-arrest-trends-caitlin-patler-2025